<a href="https://colab.research.google.com/github/Man123asa/AIA/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RAG ON COMPILER TEXTBOOK

In [ ]:
#LOCAL PIPELINE COMPILATION WITH NATIVE TEXT PARSING
import os
import torch
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline

pdf_path="aho_compilers.pdf"

print("Loading local embedding engine...")
local_embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device':'cuda' if torch.cuda.is_available() else 'cpu'}
)

print("Parsing textbook text layout page-by-page...")
doc=fitz.open(pdf_path)
full_text=[]

for page_num in range(len(doc)):
    page=doc.load_page(page_num)
    # Using explicit string-text extraction to bypass coordinate tuple variations
    page_text=page.get_text("text")
    if page_text.strip():
        full_text.append(page_text)

compiled_book_text="\n\n".join(full_text)
print(f"Extraction complete. Read {len(compiled_book_text)} total characters.")

# Slicing configurations
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", " ", ""])
book_chunks=text_splitter.split_text(compiled_book_text)
print(f"Generated {len(book_chunks)} independent text chunks.")

# Building highly reliable local FAISS database index arrays
print("Indexing chunks into vector database storage...")
vector_index=FAISS.from_texts(texts=book_chunks, embedding=local_embeddings)
retriever=vector_index.as_retriever(search_kwargs={"k":4})

if 'local_llm' not in globals():
    print("Loading language model weights into device memory...")
    model_id="Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer=AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)
    model=AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

    # Passing parameter overrides inside a distinct generation dictionary removes terminal warning screens
    generation_config={
        "max_new_tokens":512,
        "temperature":0.0,
        "do_sample":False,
        "return_full_text":False,
        "clean_up_tokenization_spaces":False
    }

    hf_pipe=pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        **generation_config
    )
    local_llm=HuggingFacePipeline(pipeline=hf_pipe)

print("Setup completed successfully.")

Loading local embedding engine...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Parsing textbook text layout page-by-page...
Extraction complete. Read 4003 total characters.
Generated 6 independent text chunks.
Indexing chunks into vector database storage...
Loading language model weights into device memory...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Setup completed successfully.


/tmp/ipykernel_591/3404806156.py:64: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  local_llm=HuggingFacePipeline(pipeline=hf_pipe)


In [ ]:
import os
import torch
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline

# Step 3: DYNAMIC EVALUATION CONSOLE TERMINAL
print("Compiler System Terminal Loop Started.")
print("Type your textbook question and hit Enter. Enter 'exit' to terminate.\n")

system_instruction=(
    "You are a precise academic assistant that answers questions based ONLY on the provided context blocks from a Compiler Design textbook.\n"
    "Look through the text context carefully:\n"
    "1. Dont confuse with the question formation and avoid the words like who, what, how, when, where"
    "2. If the context contains definitions, steps, or properties matching the question, write a complete explanation.\n"
    "3. If the context blocks do not contain the answer, or if the question asks about unmentioned topics which are NOT about compiler design and not from the Aho textbook, you must reply with exactly this sentence:\n"
    "I do not know the answer because it is not mentioned in the provided compiler textbook context.\n"
    "Be highly factual. Do not make assumptions or include any outside general training information."
)

while True:
    user_query=input("Enter your textbook question: ").strip()

    if user_query.lower() in ['exit', 'quit', 'q']:
        print("\nSession killed.")
        break

    if not user_query:
        print("Empty input. Enter a valid question.")
        continue

    print(f"Querying storage layout for: '{user_query}'...")

    retrieved_docs=retriever.invoke(user_query)
    context_text="\n---\n".join([doc.page_content for doc in retrieved_docs])

    user_message_content=f"Context:\n{context_text}\n\nQuestion:\n{user_query}"

    messages=[
        {"role":"system", "content":system_instruction},
        {"role":"user", "content":user_message_content}
    ]

    native_rag_prompt=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output=local_llm.invoke(native_rag_prompt)

    print("\nAcademic Result Summary:")
    print("-" * 50)
    print(output.strip())
    print("-" * 50 + "\n")


Compiler System Terminal Loop Started.
Type your textbook question and hit Enter. Enter 'exit' to terminate.

Enter your textbook question: What is a compiler


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Querying storage layout for: 'What is a compiler'...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Academic Result Summary:
--------------------------------------------------
A compiler is a computer program that translates source code written in a high-level programming language into machine code that can be executed directly by a computer's processor. It performs several key tasks including lexical analysis, syntax analysis, semantic analysis, intermediate code generation, optimization, and code generation. The output of a compiler is typically executable code for a specific target architecture such as x86, ARM, or MIPS. Compilers play a crucial role in software development by enabling programmers to write programs in human-readable languages while ensuring they run efficiently on computers.
--------------------------------------------------

Enter your textbook question: what is LL1 method in compiler


...

Querying storage layout for: 'Define the difference between a lexeme, a token, and a pattern.'...

Academic Result Summary:
--------------------------------------------------
In the context provided, there is no explicit definition given for "lexeme," "token," and "pattern." The text discusses various aspects of an educational institution such as its governance, academics, admissions, student services, and contact details. Therefore, I cannot provide a complete explanation based solely on the given context.

I do not know the answer because it is not mentioned in the provided compiler textbook context.
--------------------------------------------------

Enter your textbook question: what is a lexeme


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Querying storage layout for: 'what is a lexeme'...

Academic Result Summary:
--------------------------------------------------
In the given context, there is no explicit definition of "lexeme" related to compiler design. Therefore, I do not know the answer because it is not mentioned in the provided compiler textbook context.
--------------------------------------------------

Enter your textbook question: what is web development


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Querying storage layout for: 'what is web development'...

Academic Result Summary:
--------------------------------------------------
Web development refers to the creation of interactive websites using programming languages such as HTML, CSS, JavaScript, and frameworks like React, Angular, or Vue.js. It involves designing user interfaces, creating dynamic content, managing databases, and ensuring compatibility across different devices and browsers. Web developers also work on server-side technologies for backend functionality and may use tools like Node.js for serverless architectures. The goal is to build scalable, responsive, and secure applications accessible via the internet.
--------------------------------------------------

Enter your textbook question: q

Session killed.
